# Cleaned dataset (Fitzpatrick17k-C)

Filter the local Fitzpatrick17k images (and the Colab-side SAM2-segmented set) down to the **Fitzpatrick17k-C** keep list from Abhishek et al. ([Zenodo 12739457](https://doi.org/10.5281/zenodo.11101337)).

All steps below shell out to `src/clean_dataset.py`, which:
- auto-downloads `dataset/Fitzpatrick17k-C.csv` if it isn't already there,
- writes a filtered metadata CSV `dataset/fitzpatrick17k_c.csv` with the canonical `partition` (train/val/test) and `fst_consensus` columns merged in,
- moves any image whose `<md5>.jpg` filename isn't on the keep list into a `_removed/` subdirectory inside that image dir.

Run notebook from the repo root.

## 1. Local dry-run (no files moved)

Reports how many images would be removed from each local image dir and how many keep-list md5s aren't present locally. Safe to re-run.

In [2]:
!ls

Cleaned_dataset.ipynb                 cgan_architecture.png
Pipeline_test_with_10_more_imgs.ipynb colab_training.ipynb
auto_segmentation_test.ipynb          data_exploration.ipynb
baseline_inference.ipynb              our_architecture.png
baseline_metrics.ipynb                overview.ipynb


In [6]:
!python ../src/clean_dataset.py \
    --source_csv ../dataset/fitzpatrick17k_cleaned.csv \
    --output_csv ../dataset/fitzpatrick17k_c.csv \
    --image_dirs ../dataset/images \
    --mode dryrun

Source CSV rows:     15956
Keep-list rows:      11394
Filtered output:     11058
In keep but missing
  from source CSV:   336
  examples: ['0041c7551e027def40cf81ab8fe14a36', '016b658557ad43d6cc55ab2944e98e26', '01799c46d7fa9c08d3353df0c67269d8', '01be7f7454385c1abaa9d10aabcaa751', '0274d0b67e1ed7079ea95b5dedb8a5f0']
Wrote ../dataset/fitzpatrick17k_c.csv

[../dataset/images]
  files scanned:        16518
  kept (in Fitz17k-C):  11352
  to remove:            5166
  keep-set md5s missing
    from this dir:      42

Done.


## 2. Apply locally (originals + partial SAM2 set)

Same command, `--mode move`. Non-keep images go to `dataset/images/_removed/` and `dataset/images_sam2_color/_removed/` so the operation is reversible.

In [9]:
!python ../src/clean_dataset.py \
    --source_csv ../dataset/fitzpatrick17k_cleaned.csv \
    --output_csv ../dataset/fitzpatrick17k_c.csv \
    --image_dirs ../dataset/images/ \
    --mode move

Source CSV rows:     15956
Keep-list rows:      11394
Filtered output:     11058
In keep but missing
  from source CSV:   336
  examples: ['0041c7551e027def40cf81ab8fe14a36', '016b658557ad43d6cc55ab2944e98e26', '01799c46d7fa9c08d3353df0c67269d8', '01be7f7454385c1abaa9d10aabcaa751', '0274d0b67e1ed7079ea95b5dedb8a5f0']
Wrote ../dataset/fitzpatrick17k_c.csv

[../dataset/images/]
  files scanned:        11352
  kept (in Fitz17k-C):  11352
  to remove:            0
  keep-set md5s missing
    from this dir:      42

Done.


## 3. Apply on Colab to the segmented images

On Colab the SAM2-segmented dataset lives in Drive. Mount Drive, `cd` into the repo, then run the same script — pointing `--image_dirs` at the Drive path and skipping the CSV merge with `--source_csv ""` (the filtered CSV is already in this repo).

Edit `SEG_DIR` to match where your segmented images actually live.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SkinLesionBiasReduction

Mounted at /content/drive
/content/drive/MyDrive/SkinLesionBiasReduction


In [12]:
!git stash --include-untracked

Saved working directory and index state WIP on main: 1feb687 Add segmentation


In [13]:
!git pull origin main

From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
Updating 1feb687..61c4a6b
Fast-forward
 .DS_Store                                          |  Bin 14340 -> 14340 bytes
 .gitignore                                         |    3 +-
 ...tology_with_the_Fitzpatrick_17k_Dataset (1).pdf | 9081 ++++++++++++++++++++
 dataset/.DS_Store                                  |  Bin 10244 -> 8196 bytes
 logs/20260427_055032_report.md                     |   77 +
 notebooks/Cleaned_dataset.ipynb                    |  211 +
 notebooks/colab_training.ipynb                     | 1365 ++-
 plan_to_clean_dataset.md                           |   95 +
 src/baseline_metrics.py                            |   10 +-
 src/clean_dataset.py                               |  186 +
 10 files changed, 10806 insertions(+), 222 deletions(-)
 create mode 100644 Evaluating_Deep_Neural_Networks_Trained_on_Clinical_Images_in_Dermatology_with_the_Fitzpatrick_17k_Dataset (

In [11]:
!git fetch origin main && git pull -f

From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
Updating 1feb687..61c4a6b
error: Your local changes to the following files would be overwritten by merge:
	logs/20260427_055032_report.md
Please commit your changes or stash them before you merge.
Aborting


In [22]:
# --- Colab only ---
   # path to this repo on Drive

SEG_DIR = "/content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_sam2_224"

!python src/clean_dataset.py \
    --keep_csv dataset/Fitzpatrick17k-C.csv \
    --source_csv "" \
    --image_dirs "$SEG_DIR" \
    --mode move

Loaded 11394 md5 hashes from dataset/Fitzpatrick17k-C.csv

[/content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_sam2_224]
  files scanned:        11058
  kept (in Fitz17k-C):  11058
  to remove:            0
  keep-set md5s missing
    from this dir:      336

Done.


## 4. Verify the cleaned CSV

Quick sanity check on the filtered metadata: row counts, partition split, and class/FST distribution.

In [ ]:
import pandas as pd

df = pd.read_csv("dataset/fitzpatrick17k_c.csv")
print("rows:", len(df))
print("\npartition:")
print(df["partition"].value_counts())
print("\nthree_partition_label:")
print(df["three_partition_label"].value_counts())
print("\nfitzpatrick_scale:")
print(df["fitzpatrick_scale"].value_counts().sort_index())